In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

league = pd.read_csv('../../data/league_weather_2021_2025.csv')
heatmap = pd.read_csv('../../data/heatmap_data_all_stadiums.csv')

bos = league[league['home_team'] == 'BOS'].copy()
bos['game_date'] = pd.to_datetime(bos['game_date'])
bos['month'] = bos['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
bos['month_name'] = bos['month'].map(month_map)
bos['rhum_bin'] = pd.qcut(bos['rhum'], q=5, duplicates='drop')

print(f'Total BOS home games: {len(bos)}')
print('\nHumidity bins and game counts:')
print(bos['rhum_bin'].value_counts().sort_index())

In [ ]:
# Cross-Stadium Comparison: How Unusual Is Fenway's Humidity Effect?
away_runs_humidity = heatmap[(heatmap['metric'] == 'away_runs') & (heatmap['weather_variable'] == 'rhum')].copy()

stadium_ranges = away_runs_humidity.groupby('team_code').agg(
    max_diff=('diff_from_mean', 'max'),
    min_diff=('diff_from_mean', 'min')
).reset_index()
stadium_ranges['effect_range'] = stadium_ranges['max_diff'] - stadium_ranges['min_diff']
stadium_ranges = stadium_ranges.sort_values('effect_range', ascending=False).reset_index(drop=True)
stadium_ranges['rank'] = stadium_ranges.index + 1

bos_rank = stadium_ranges[stadium_ranges['team_code'] == 'BOS']
print('Fenway rank by humidity effect range on away runs:')
display(bos_rank)

print('Top 10 stadiums by humidity effect range on away runs:')
display(stadium_ranges.head(10))

In [ ]:
# Fenway-Specific Comparison Across Weather Variables
bos_weather_strength = heatmap[(heatmap['team_code'] == 'BOS') & (heatmap['metric'] == 'away_runs')].copy()
bos_weather_strength = bos_weather_strength.groupby('weather_variable').agg(
    max_diff=('diff_from_mean', 'max'),
    min_diff=('diff_from_mean', 'min')
).reset_index()
bos_weather_strength['effect_range'] = bos_weather_strength['max_diff'] - bos_weather_strength['min_diff']
bos_weather_strength = bos_weather_strength.sort_values('effect_range', ascending=False).reset_index(drop=True)
bos_weather_strength

In [ ]:
# Fenway Humidity Quintiles: Summary Table
humidity_summary = bos.groupby('rhum_bin', observed=True).agg(
    games=('away_runs_scored', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    total_runs_mean=('total_runs', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    avg_exit_velocity_mean=('avg_exit_velocity', 'mean'),
    n_barrels_mean=('n_barrels', 'mean'),
    barrel_rate_mean=('barrel_rate', 'mean'),
    hr_h_ratio_mean=('hr_h_ratio', 'mean'),
    temp_mean=('temp_f', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(3)

humidity_summary.index.name = 'Humidity Bin (%)'
humidity_summary.columns = [
    'Games', 'Away Runs Mean', 'Total Runs Mean', 'Hits Mean', 'Home Runs Hit Mean',
    'Avg Exit Velocity', 'Barrels Mean', 'Barrel Rate', 'HR:H Ratio',
    'Temp Mean', 'Pressure Mean', 'Wind Mean'
]
humidity_summary

In [ ]:
# Fenway Humidity Profile: Runs, Contact, and Co-Moving Weather
bin_order = bos['rhum_bin'].cat.categories
tick_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in bin_order]
x = np.arange(len(bin_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(x, bos.groupby('rhum_bin', observed=True)['away_runs_scored'].mean().reindex(bin_order), color='#d62728', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Away Runs')
axes[0, 0].set_title('Away Runs by Humidity Quintile')
axes[0, 0].yaxis.grid(True, alpha=0.3)

bar_width = 0.35
hits_means = bos.groupby('rhum_bin', observed=True)['hits'].mean().reindex(bin_order)
hr_means = bos.groupby('rhum_bin', observed=True)['home_runs_hit'].mean().reindex(bin_order)
axes[0, 1].bar(x - bar_width / 2, hits_means, width=bar_width, color='#1f77b4', alpha=0.8, label='Hits', edgecolor='black', linewidth=0.5)
axes[0, 1].bar(x + bar_width / 2, hr_means, width=bar_width, color='#ff7f0e', alpha=0.8, label='Home Runs Hit', edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 1].set_title('Hits and Home Runs by Humidity Quintile')
axes[0, 1].legend()
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].plot(x, bos.groupby('rhum_bin', observed=True)['avg_exit_velocity'].mean().reindex(bin_order), marker='o', color='#2ca02c', linewidth=2)
axes[1, 0].plot(x, bos.groupby('rhum_bin', observed=True)['barrel_rate'].mean().reindex(bin_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 0].set_title('Exit Velo and Barrel Rate by Humidity Quintile')
axes[1, 0].legend(['Avg Exit Velocity', 'Barrel Rate'])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, bos.groupby('rhum_bin', observed=True)['temp_f'].mean().reindex(bin_order), marker='o', color='#d62728', linewidth=2)
axes[1, 1].plot(x, bos.groupby('rhum_bin', observed=True)['pres'].mean().reindex(bin_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 1].plot(x, bos.groupby('rhum_bin', observed=True)['wspd_mph'].mean().reindex(bin_order), marker='o', color='#1f77b4', linewidth=2)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 1].set_title('Temp, Pressure, and Wind by Humidity Quintile')
axes[1, 1].legend(['Temperature', 'Pressure', 'Wind'])
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Fenway: What Changes Across Humidity Quintiles?', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Driest Quintile vs. Everyone Else
lowest_bin = bos['rhum_bin'].cat.categories[0]
bos['dry_group'] = np.where(bos['rhum_bin'] == lowest_bin, 'Driest Quintile', 'Other Quintiles')

dry_summary = bos.groupby('dry_group').agg(
    games=('away_runs_scored', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    total_runs_mean=('total_runs', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    avg_exit_velocity_mean=('avg_exit_velocity', 'mean'),
    barrel_rate_mean=('barrel_rate', 'mean'),
    temp_mean=('temp_f', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(3)
dry_summary

In [ ]:
# Seasonal Context: Humidity and Away Runs by Month
months = sorted(bos['month'].dropna().unique())
month_names = [month_map.get(m, str(m)) for m in months]
x = np.arange(len(months))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

month_humidity = bos.groupby('month')['rhum'].mean().reindex(months)
month_runs = bos.groupby('month')['away_runs_scored'].mean().reindex(months)

axes[0].plot(x, month_humidity, marker='o', color='#1f77b4', linewidth=2)
axes[0].set_xticks(x)
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Humidity (%)')
axes[0].set_title('Average Humidity by Month')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, month_runs, marker='o', color='#d62728', linewidth=2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Away Runs')
axes[1].set_title('Average Away Runs by Month')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Fenway: Does Humidity Align with Seasonal Scoring Swings?', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()